# LSTM Model Multivariate


n this section we implement multivariate forecasting using the LSTM Model (Long Short-Term Memory) with the **TimeSeriesDatasetVectorizedExog** approach.

The LSTM (Long Short-Term Memory) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

The model architecture  remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. The LSTM's gating mechanisms effectively capture long-term temporal dependencies across all series.

**Layer Breakdown:**

- **LSTM Layers**: 2 stacked LSTM layers with memory cells and three gates (input, forget, output)
- **Hidden Size**: 128 units per layer (default)
- **Dropout**: Applied between LSTM layers (if >1 layer) and before final output
- **Output Layer**: Single fully connected layer producing 1-step forecast
- **Input Features**: Value + GDP + CPI + Interest_Rate + Year + Month + One-Hot Encoding (1502 dims)

In [ ]:
import torch
import torch.nn as nn

## Model

In [ ]:
class LSTMForecaster(nn.Module):
    """
    LSTM model for MULTIVARIATE time series forecasting.
    Architecture: LSTM -> Dropout -> LSTM -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: LSTM hidden dimension
            num_layers: Number of LSTM layers
            dropout: Dropout rate
        """
        super(LSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Take the output from the last time step
        last_output = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

## Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|--------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.1502 | 4 | 0.487 | 128 | 0.000298 | 2 | 9 |
| 1 | 0.1382 | 16 | 0.353 | 64 | 0.001897 | 1 | 1 |
| 2 | 0.1550 | 8 | 0.258 | 128 | 0.000121 | 1 | 3 |
| 3 | 0.1222 | 16 | 0.310 | 256 | 0.007550 | 1 | 9 |
| 4 | 0.1285 | 16 | 0.244 | 128 | 0.004533 | 1 | 4 |


### Best Hyperparameters

Best Hyperparameters for Trial 3:
- learning_rate: 0.0075
- batch_size: 16
- num_layers: 1
- hidden_size: 256
- dropout: 0.301

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/lstm/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/lstm/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/lstm/fold3/fold_results.png)


### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|---------|---------|---------|-----------|
| Fold 1 | 60917.25 | 246.81 | 105.27 | 0.8793 | 59.96 |
| Fold 2 | 52741.32 | 229.65 | 100.43 | 0.8835 | 69.61 |
| Fold 3 | 48601.75 | 220.46 | 91.48 | 0.9001 | 53.53 |
| **Average** | **54086.77 ± 6267.02** | **232.31 ± 13.38** | **99.06 ± 7.00** | **0.8876 ± 0.0110** | **61.03 ± 8.10** |

### SMAPE Distribution Accross Folds

| SMAPE Range | Percentage of Series | Average Number of Series |
|-------------|---------------------|-------------------------|
| <10% | 15.8% ± 2.1% | 238 |
| 10-20% | 10.5% ± 1.8% | 157 |
| 20-30% | 12.0% ± 3.4% | 181 |
| 30-40% | 9.8% ± 1.6% | 148 |
| >40% | 51.9% ± 6.0% | 779 |


**Comparison with Baseline:**

The LSTM multivariate model achieves an average SMAPE of 61.03% ± 8.10%, which is **11.23 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). The LSTM model demonstrates superior performance with more consistent results across folds, showing comparable variation (standard deviation: 8.10% vs 7.06%). The model performs particularly well in Fold 3 (53.53% SMAPE), indicating its ability to effectively capture complex temporal dependencies. This improvement over the baseline suggests that the LSTM architecture successfully leverages its memory mechanisms to model long-term patterns in the multivariate time series data.

## Model Results with Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|--------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.2097 | 4 | 0.387 | 64 | 0.002841 | 1 | 2 |
| 1 | 0.4520 | 16 | 0.376 | 32 | 0.000857 | 1 | 0 |
| 2 | 0.3524 | 4 | 0.335 | 128 | 0.000189 | 1 | 4 |
| 3 | 0.4123 | 16 | 0.420 | 256 | 0.000123 | 3 | 39 |
| 4 | 0.5338 | 16 | 0.193 | 64 | 0.000101 | 2 | 4 |

### Best Hyperparameters 

Parameters from Trial 0:
- learning_rate: 0.0028
- batch_size: 4
- num_layers: 1
- hidden_size: 64
- dropout: 0.387

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/lstm_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/lstm_exog/fold2/fold_results.png)


#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30
    
![Fold 3 Results](./img/multivariate/lstm_exog/fold3/fold_results.png)


### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|----------|---------|---------|---------|-----------|
| Fold 1 | 192022.05 | 438.20 | 208.26 | 0.6195 | 126.61 |
| Fold 2 | 90326.60 | 300.54 | 193.82 | 0.8005 | 116.41 |
| Fold 3 | 67391.33 | 259.60 | 115.98 | 0.8614 | 101.07 |
| **Average** | **116579.99 ± 66333.51** | **332.78 ± 93.56** | **172.69 ± 49.64** | **0.7605 ± 0.1259** | **114.70 ± 12.85** |

### Average SMAPE Distribution Accross Folds 

| SMAPE Range | Percentage of Series | Average Number of Series |
|-------------|---------------------|-------------------------|
| <10% | 7.3% ± 7.3% | 109 |
| 10-20% | 7.9% ± 6.7% | 118 |
| 20-30% | 6.5% ± 5.1% | 97 |
| 30-40% | 5.6% ± 3.8% | 84 |
| >40% | 72.8% ± 9.4% | 1093 |

**Comparison with Baseline:**

The LSTM multivariate model with exogenous features achieves an average SMAPE of 114.70% ± 12.85%, which is **42.44 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). The model shows significantly worse performance with higher variation (standard deviation: 12.85% vs 7.06%). While the model shows improvement from Fold 1 (126.61%) to Fold 3 (101.07%), it consistently underperforms the baseline across all folds. The high error rates suggest that the addition of exogenous features (GDP, CPI, Interest Rate) may have introduced noise or that the model architecture struggles to effectively integrate these additional variables. This indicates that the current configuration does not successfully leverage the exogenous information, and the LSTM model without exoenous variable or even alternative architectures may be more appropriate for this dataset.